In [1]:
!pip install sru==3.0.0.dev6

## All imports

In [2]:
# For Driver code
import glob
import random
import torch
import os

# for Models
import torch
import torch.nn as nn
from torchvision import models
from sru import SRUpp
from torch.nn import Module, Dropout, Linear

# For utilities
import cv2
import numpy as np

## Utility functions

- Two functions, `get_frames()` returns all the frames of a video in array-like format
- `get_numeric_label()` returns a numerical value with a given class label

In [3]:
# Function for getting all frames
def get_frames(path):
    video = cv2.VideoCapture(path)
    if not video.isOpened():
        raise IOError("Error reading video file")

    frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []
    for i in range(0, frame_count):
        ret, frame = video.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))
            frame = frame.astype("float32") / 255.0
            frames.append(frame)
    video.release()
    return np.array(frames)

# Function for getting numerical frames
def get_numeric_label(category):
    if category == 'abuse':
        return 1
    elif category == 'arson':
        return 2
    elif category == 'assault':
        return 3
    elif category == 'roadaccidents':
        return 4
    elif category == 'burglary':
        return 5
    elif category == 'explosion':
        return 6
    elif category == 'fighting':
        return 7
    elif category == 'robbery':
        return 8
    elif category == 'shooting':
        return 9
    elif category == 'stealing':
        return 10
    elif category == 'vandalism':
        return 11
    else:
        return 0

## Models

This section contains two models
- `Resnet` model which is used to generate embeddings
- `SRU` model which classifies a given video

In [4]:
# Set the device to use (e.g., 'cpu', 'cuda', 'mps')
device = torch.device("mps" if torch.backends.mps.is_available() else
                      ("cuda" if torch.cuda.is_available() else "cpu"))

# Select Device According to Availability
print("Device selected:", device)

# If the device is CUDA, print the device capability
if device.type == "cuda":
    os.system("nvidia-smi")
    print()
    print("Device type:", device.type)
    print("Capability:", torch.cuda.get_device_capability(device))
else:
    print("Device capabilities are limited on MPSs and CPUs.")

Device selected: cuda
Mon May 27 11:36:29 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:00:04.0 Off |                    0 |
| N/A   30C    P0              25W / 250W |      2MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+---

In [5]:
# Resnet Model
class ResNetAnomaly(nn.Module):
    def __init__(self):
        super(ResNetAnomaly, self).__init__()

        # Load the pre-trained ResNet50 model without fine-tuning
        self.resnet = models.resnet50(weights=None)
        self.resnet.load_state_dict(torch.load('/kaggle/input/test-models/resnet50.pth'))

        # Get the number of features from the last layer of the ResNet
        num_ftrs = self.resnet.fc.in_features

        # Modify the last layer to output 2048 features
        self.resnet.fc = nn.Linear(num_ftrs, 2048)

        # Define additional layers for feature learning
        self.additional_layers = nn.Sequential(
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
        )

        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(2048, 2048),
            nn.Tanh(),
            nn.Linear(2048, 1),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        x = x.to(device)
        x = self.resnet(x)

        # Attention mechanism
        attention_weights = self.attention(x)
        attended_features = torch.matmul(attention_weights.unsqueeze(dim=2), x.unsqueeze(dim=1)).squeeze(dim=1)

        # Pass the attended features through the additional layers for further learning
        x = self.additional_layers(attended_features)

        return x

In [6]:
# SRUpp Model
class SRUppModel(Module):
    def __init__(self, input_size, hidden_size, **kwargs):
        super(SRUppModel, self).__init__()
        # Main SRU layer
        self.sru_layers = SRUpp(
            proj_size=2048,
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=kwargs.get('num_layers', 2),
            dropout=kwargs.get('dropout_prob', 0.0),
            bidirectional=kwargs.get('bidirectional', False),
            layer_norm=kwargs.get('layer_norm', True),
            highway_bias=kwargs.get('highway_bias', 0.0),
            rescale=kwargs.get('rescale', True),
            nn_rnn_compatible_return=kwargs.get('nn_rnn_compatible_return', False),
            proj_input_to_hidden_first=kwargs.get('proj_input_to_hidden_first', False),
            normalize_after=kwargs.get('normalize_after', False),
        ).to(device)
        # Dropout layer
        self.dropout = Dropout(kwargs.get('dropout_layer_prob', 0.2)).to(device)
        # Linear layer (Fully connected layer)
        self.linear = Linear(
            hidden_size,
            out_features=kwargs.get('num_classes', 12)
        ).to(device)
        # L2 regularization
        self.l2_reg_lambda = kwargs.get('l2_reg_lambda', 1e-5)

    def forward(self, x):
        output_states, _, _ = self.sru_layers(x)
        output = self.linear(self.dropout(output_states[-1]))
        return output, output_states

    def l2_regularization(self):
        l2_reg = torch.tensor(0., device=device)
        for param in self.parameters():
            l2_reg += torch.norm(param, p=2)
        return self.l2_reg_lambda * l2_reg

In [7]:
# Instantiate the models
modelResnet = ResNetAnomaly()
modelResnet.to(device)
modelResnet.eval()

ResNetAnomaly(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
     

In [8]:
modelSRU = SRUppModel(2048, 1024)
modelSRU = modelSRU.to(device)
modelSRU.eval()
# Set weights and biases for SRU
modelSRU.load_state_dict(torch.load('/kaggle/input/test-models/sru.pth'))

<All keys matched successfully>

# DRIVER CODE

## Get all the videos

In [9]:
# Get all video paths
video_path = '/kaggle/input/test-dataset-v3/*/*.mp4'
video_path = glob.glob(video_path)
random.shuffle(video_path)

## Loop through each video and get predicted labels

In [10]:
total_correct = 0
total_test_case = len(video_path)

In [11]:
import statistics
for video in video_path:
    print(f"Current video path: {video}")
    frames = get_frames(video)
    # print(f"Total video shape: {frames.shape}")

    # Determine ground truth label
    label = get_numeric_label(video.split("/")[4])
    print(f"Current video label: {label}")

    # Determine Segments, each segment will have 100 frames (10 seconds)
    segments = frames.shape[0] // 100
    print(f"Total Segments: {segments}")
    
    # Now we have the segments of the video
    # Create segment array list
    segment_list = []
    for segment in range(segments):
        # print(f"\t\tStart point {segment * 100} End point {(segment + 1) * 100}")
        s = frames[segment * 100:(segment + 1) * 100]
        segment_list.append(s)
        del s
        
    del frames
    del segments
        
    # print(f"Asserting total number of segments: {len(segment_list)}")
    
    
    predict_list = []
    class_counts = {i: 0 for i in range(12)}
    
    
    # Get the embeddings
    for s in segment_list:
        # print(f"Segment Shape: {s.shape}")
        # Reshape the segment
        s = torch.tensor(s).permute(0, 3, 1, 2).to(device)
        # print(f"Segment Shape after reshaping: {s.shape}")
        # Feed the segment to the model
        embeddings = modelResnet(s)
        del s
#         print(f"Embedding Shape: {embeddings.shape}")

        # Prepare the shape for SRU
        embeddings = embeddings.unsqueeze(0)
        em = embeddings.permute(1, 0, 2)
#         print(f"Embedding Shape after reshaping: {em.shape}")
        del embeddings
        
        # SRU
        
        # Get the prediction
        with torch.no_grad():
            outputs, _ = modelSRU(em)
            del em
            _, predicted = torch.max(outputs.data, 1)
            predicted_label = predicted.item()
            predict_list.append(predicted.item())
            del predicted
            print(f"Predicted Label: {predicted_label}")

            # Increment the count of the predicted class
            class_counts[predicted_label] += 1
            
            del _
            
            
#   print(predict_list)

    total_predictions = sum(class_counts.values())
    for class_label, count in class_counts.items():
        percentage = (count / total_predictions) * 100
        print(f"Class {class_label}: {percentage}%")


    # Take max value
    p = statistics.mode(predict_list)
    print(f"Predicted Label: {p}")
    if p == label:
        total_correct = total_correct + 1
        
    del predict_list
        
    del segment_list
    print("\n")

Current video path: /kaggle/input/test-dataset-v3/vandalism/Vandalism017_x264.mp4
Current video label: 11
Total Segments: 10
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Predicted Label: 0
Class 0: 100.0%
Class 1: 0.0%
Class 2: 0.0%
Class 3: 0.0%
Class 4: 0.0%
Class 5: 0.0%
Class 6: 0.0%
Class 7: 0.0%
Class 8: 0.0%
Class 9: 0.0%
Class 10: 0.0%
Class 11: 0.0%
Predicted Label: 0


Current video path: /kaggle/input/test-dataset-v3/assault/Assault025_x264.mp4
Current video label: 3
Total Segments: 11
Predicted Label: 0
Predicted Label: 5
Predicted Label: 0
Predicted Label: 5
Predicted Label: 0
Predicted Label: 5
Predicted Label: 5
Predicted Label: 5
Predicted Label: 0
Predicted Label: 5
Predicted Label: 5
Class 0: 36.36363636363637%
Class 1: 0.0%
Class 2: 0.0%
Class 3: 0.0%
Class 4: 0.0%
Class 5: 63.63636363636363%
Class 6: 0.0%
Class 7: 0.0%
Class 8: 0.0%
Class 9: 

In [12]:
accuracy = (total_correct/total_test_case)*100
print(f"Accuracy: {accuracy}")

Accuracy: 47.61904761904761
